# 01 · O nó de coleta: o que um ESP32 diz sobre uma máquina

**Objetivo:** entender o que o nó lê da máquina, o que ele publica no MQTT e por que três firmwares escritos em momentos diferentes acabaram falando três dialetos.

## Antes de começar

- **Você precisa saber:** o básico de MQTT (tópico, publicar, assinar). Se não souber, a lição 05 do [iiot-protocols-examples](https://github.com/MatheusEduGarcia/iiot-protocols-examples) cobre isso.
- **Você não precisa de:** hardware. O `oeelab.node.Node` é o ESP32 sem a placa: mesmas entradas, mesmas mensagens.
- **Tempo:** 20 minutos.
- **Broker:** só a seção 4 usa um broker em `127.0.0.1:1883`. O resto roda sem rede.

![Arquitetura](../docs/img/architecture.png)

In [1]:
import sys, json, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
%matplotlib inline

## 1. O que o nó lê

Um nó, uma máquina, até cinco sinais:

| Entrada | De onde vem | Tipo | Vira |
|---|---|---|---|
| start | botão ou sinal de ciclo | borda | evento `start` |
| stop | botão ou sinal de parada | borda | evento `stop` |
| peça boa | sensor ótico ou indutivo na saída | pulso | contador `output-good` |
| refugo | botão do operador | pulso | contador `output-defect` |
| rodando | contato auxiliar do contator | nível | `state` |

![Ligação](../docs/img/wiring.png)

O firmware usa `INPUT_PULLUP` com nível baixo ativo. Na bancada basta um botão entre o pino e o GND. Na máquina, sinais de 24 V **nunca** vão direto no pino: passam por um estágio optoisolado. Os detalhes estão em [docs/hardware.md](../docs/hardware.md).

## 2. O que o firmware de hoje publica (v0)

Vamos ligar um nó virtual no dialeto `v0`, o do firmware `esp32_oee_full_signals`, e capturar o que sai. O parâmetro `wire` entrega as mensagens a uma função em vez de a um broker.

In [2]:
from datetime import datetime, timedelta, timezone
from oeelab.contract import Machine
from oeelab.node import Node

maquina = Machine("packaging", "line-1", "cell-a", "filler-01")
t0 = datetime(2026, 11, 10, 9, 0, 0, tzinfo=timezone.utc)

capturado = []
no = Node(maquina, "v0", wire=lambda topico, payload, retido: capturado.append((topico, payload.decode(), retido)))
no.connect(t0)
no.press_start(t0 + timedelta(seconds=1))
no.piece(t0 + timedelta(seconds=3))
no.piece(t0 + timedelta(seconds=5), good=False)
no.press_stop(t0 + timedelta(seconds=9))

for topico, payload, retido in capturado:
    print(f"{topico.split('/')[-1]:14s} {'R' if retido else ' '}  {payload}")

status         R  {"online":true,"ip":"192.168.0.50","rssi":-61,"client":"node-filler-01-344848"}
start             {"ts":"2026-11-10T09:00:01","value":1,"machine_id":"FILLER-01"}
state          R  {"ts":"2026-11-10T09:00:01","value":1,"running":true,"machine_id":"FILLER-01"}
output-good       {"ts":"2026-11-10T09:00:03","value":1,"machine_id":"FILLER-01"}
output-defect     {"ts":"2026-11-10T09:00:05","value":1,"machine_id":"FILLER-01"}
stop              {"ts":"2026-11-10T09:00:09","value":1,"machine_id":"FILLER-01"}
state          R  {"ts":"2026-11-10T09:00:09","value":0,"running":false,"machine_id":"FILLER-01"}


Leia o payload de `output-good` com calma. Ele tem um horário com resolução de **um segundo**, o valor `1` e o nome da máquina. Mais nada.

### Pare e preveja

A máquina faz **duas peças dentro do mesmo segundo**. O nó publica uma mensagem por peça. O que diferencia a primeira mensagem da segunda?

Decida antes de rodar a próxima célula.

In [3]:
capturado.clear()
no.piece(t0 + timedelta(seconds=20, milliseconds=100))
no.piece(t0 + timedelta(seconds=20, milliseconds=600))
a, b = [p for _, p, _ in capturado]
print(a); print(b)
print("\nAs duas mensagens são idênticas byte a byte?", a == b)

{"ts":"2026-11-10T09:00:20","value":1,"machine_id":"FILLER-01"}
{"ts":"2026-11-10T09:00:20","value":1,"machine_id":"FILLER-01"}

As duas mensagens são idênticas byte a byte? True


Nada diferencia. Para quem recebe, isso é indistinguível de **uma** mensagem entregue duas vezes, coisa que o MQTT faz de verdade (QoS 1, bridges, reconexão). Um receptor cuidadoso descarta repetições, e aqui o cuidado vira perda de peça. O notebook 02 mede o tamanho do estrago.

## 3. Três firmwares, três dialetos

O repositório tem três sketches em `firmware/`, e cada um nasceu para resolver um caso:

In [4]:
dialetos = {
  "esp32_oee_full_signals":  ("oee/uns/packaging/line-1/cell-a/filler-01/output",   '{"ts":"2026-11-10T06:00:00","value":1,"machine_id":"FILLER-01"}'),
  "esp32_oee_webconfig":     ("fabrica/linha1/m1/input/3",                            '{"label":"Peca boa","state":1,"ts":123456}'),
  "esp32_cnc_lathe_1button": ("lab/line1/cnc_lathe/button/start",                     '{"ts":"2026-11-10T06:00:00","value":1.0,"machine_id":"CNC-001"}'),
}
for nome, (topico, payload) in dialetos.items():
    print(f"{nome}\n   tópico  {topico}\n   payload {payload}\n")

esp32_oee_full_signals
   tópico  oee/uns/packaging/line-1/cell-a/filler-01/output
   payload {"ts":"2026-11-10T06:00:00","value":1,"machine_id":"FILLER-01"}

esp32_oee_webconfig
   tópico  fabrica/linha1/m1/input/3
   payload {"label":"Peca boa","state":1,"ts":123456}

esp32_cnc_lathe_1button
   tópico  lab/line1/cnc_lathe/button/start
   payload {"ts":"2026-11-10T06:00:00","value":1.0,"machine_id":"CNC-001"}



Compare os três e anote as diferenças. As que importam:

| Ponto | full_signals | webconfig | cnc_lathe |
|---|---|---|---|
| Onde fica o nome do sinal | no tópico (`output`) | num rótulo livre dentro do payload | no tópico (`button/start`) |
| `ts` | hora **local** sem fuso, em segundos | **milissegundos desde o boot** | hora local sem fuso |
| Sem NTP | publica `1970-01-01T00:00:00` | não se aplica | publica 1970 |
| Hierarquia do tópico | `uns/area/linha/célula/máquina` | livre | `lab/line1/...` |

Nenhum está "errado" isolado. O problema aparece quando um único backend precisa entender os três, e é para isso que serve um contrato.

## 4. O nó morreu ou a máquina parou?

Sem dados chegando, o painel não sabe distinguir "máquina parada" de "nó sem energia". O MQTT resolve com o **last will**: ao conectar, o nó deixa com o broker uma mensagem para ser publicada se ele sumir.

Esta seção usa o broker de verdade.

In [5]:
import time, paho.mqtt.client as mqtt
from oeelab.node import now_utc

m = Machine("lab", "line-9", "cell-z", "demo-01", prefix="oeenb")
visto = []
ouvinte = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
ouvinte.on_message = lambda c, u, msg: msg.payload and visto.append((msg.topic.split("/")[-1], json.loads(msg.payload)))
ouvinte.connect("127.0.0.1", 1883); ouvinte.subscribe(m.base + "/status"); ouvinte.loop_start()

no_real = Node(m, "v1")
no_real.connect(now_utc()); time.sleep(0.8)
print("depois de conectar :", visto[-1][1])

no_real.disconnect(graceful=False)        # queda de energia: o socket fecha sem aviso
time.sleep(1.5)
print("depois da queda    :", visto[-1][1])

ouvinte.publish(m.base + "/status", b"", retain=True); time.sleep(0.2); ouvinte.loop_stop()

depois de conectar : {'v': 1, 'online': True, 'boot': '9c312d', 'fw': 'oee-node-sim 1.0.0', 'ip': '192.168.0.50', 'rssi': -61, 'uptime_s': 0, 'clock_synced': True, 'ts': '2026-11-17T00:16:48.535Z'}


depois da queda    : {'v': 1, 'online': False}


<MQTTErrorCode.MQTT_ERR_SUCCESS: 0>

A segunda mensagem **não foi enviada pelo nó**, que já estava morto. Foi o broker que a publicou em nome dele. Por isso ela não tem `seq`, `boot` nem `ts`: foi escrita no momento da conexão.

![Ciclo de vida](../docs/img/lifecycle.png)

## Erros comuns

- **Ligar 24 V direto no pino.** O ESP32 é 3,3 V e não tolera nem 5 V. Use optoacoplador.
- **Confiar no relógio do nó.** Sem NTP o ESP32 acorda em 1970. O firmware precisa saber que o relógio não é confiável e dizer isso.
- **Usar hora local no payload.** Quem recebe não sabe o fuso. O payload leva UTC e quem exibe converte.
- **Achar que "sem mensagem" significa "parada".** Sem last will, um nó desligado parece uma máquina muito bem comportada.
- **Deixar o nome do sinal num rótulo livre.** "Peca boa", "peça boa" e "OK" viram três sinais diferentes.

## Confira o que ficou

**1. Por que duas peças no mesmo segundo geram mensagens idênticas no v0?**

<details><summary>Ver resposta</summary>

Porque o único campo que muda entre mensagens é o `ts`, e ele tem resolução de um segundo. Valor (`1`) e máquina são sempre iguais.

</details>

**2. Quem publica a mensagem `online: false` quando o nó perde energia?**

<details><summary>Ver resposta</summary>

O broker. O nó registrou essa mensagem como last will ao conectar.

</details>

**3. O que há de errado em publicar `millis()` como `ts`?**

<details><summary>Ver resposta</summary>

É o tempo desde o boot, não um instante. Não dá para colocar o evento no calendário, e o valor volta a zero a cada reinício.

</details>

**4. Uma máquina só tem um contato de 'rodando'. Que parte do OEE dá para calcular?**

<details><summary>Ver resposta</summary>

A disponibilidade. Sem contagem de peças não há performance nem qualidade.

</details>

## Exercícios

1. Crie um nó `v0` com `clock_synced=False`, publique três peças e observe o `ts`. O que o receptor consegue concluir sobre a ordem delas?
2. Rode a seção 4 com `graceful=True`. O last will é publicado? Por quê?
3. Abra `firmware/esp32_oee_webconfig/esp32_oee_webconfig.ino`, ache `handleInputs()` e descubra o que acontece quando um contato "quica" três vezes em 10 ms.

## Glossário

- **Nó de coleta:** o ESP32 ligado aos sinais de uma máquina.
- **Borda, pulso, nível:** mudança de estado, mudança curta que vai e volta, e estado que permanece.
- **Last will:** mensagem que o broker publica em nome de um cliente que desapareceu.
- **Retido (retain):** o broker guarda a última mensagem do tópico e a entrega a quem assinar depois.
- **Optoacoplador:** componente que passa o sinal por luz, sem ligação elétrica entre os dois lados.

## Próximo passo

No notebook **02** os três problemas viram números, e o contrato v1 resolve cada um com um campo.